# **Building and Training a Feedforward Neural Network for Language Modeling**

This project explores the use of Feedforward Neural Networks (FNNs) in language modeling. The primary objective is to build a neural network that learns word relationships and generates meaningful text sequences. The implementation is done using PyTorch, covering key aspects of Natural Language Processing (NLP), such as:
* Tokenization & Indexing: Converting text into numerical representations.
* Embedding Layers: Mapping words to dense vector representations for efficient learning.
* Context-Target Pair Generation (N-grams): Structuring training data for sequence prediction.
* Multi-Class Neural Network: Designing a model to predict the next word in a sequence.

The training process includes optimizing the model with loss functions and backpropagation techniques to improve accuracy and coherence in text generation. By the end of the project, you will have a working FNN-based language model capable of generating text sequences.



# **Objectives**

After completing this lab, you will be able to:

 - Implement a feedforward neural network using the PyTorch framework, including embedding layers, for language modeling tasks.
 - Fine-tune the output layer of the neural network for optimal performance in text generation.
 - Apply various training strategies and fundamental Natural Language Processing (NLP) techniques, such as tokenization and sequence analysis, to improve text generation.


#### Import Libraries

In [1]:
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import nltk
import torch
import re
import string
from nltk.tokenize import word_tokenize


## Feedforward Neural Networks (FNNs) for language models

FNNs, or Multi-Layer Perceptrons, serve as the foundational components for comprehending neural networks in natural language processing (NLP). In NLP tasks, FNNs process textual data by transforming it into numerical vectors known as embeddings. Subsequently, these embeddings are input to the network to predict language facets, such as the upcoming word in a sentence or the sentiment of a text.

Let's consider the following song lyrics for our analysis.


In [2]:
song= """We are no strangers to love
You know the rules and so do I
A full commitments what Im thinking of
You wouldnt get this from any other guy
I just wanna tell you how Im feeling
Gotta make you understand
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Weve known each other for so long
Your hearts been aching but youre too shy to say it
Inside we both know whats been going on
We know the game and were gonna play it
And if you ask me how Im feeling
Dont tell me youre too blind to see
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Weve known each other for so long
Your hearts been aching but youre too shy to say it
Inside we both know whats been going on
We know the game and were gonna play it
I just wanna tell you how Im feeling
Gotta make you understand
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you"""

### Data Preprocessing

In [3]:
def preprocess_string(s):
    
    # Remove all non-word characters (everything except letters and numbers)
    # \w matches any word character (letters, numbers, and underscores)
    # \s matches any whitespace characters
    # ^ inside [] negates the selection, so [^\w\s] matches anything that's NOT a word character or whitespace.
    s = re.sub(r"[^\w\s]",'', s)
    
    # Remove all whitespace characters (spaces, tabs, newlines)
    # \s+ matches one or more whitespace characters.
    s = re.sub(r"[\s+]",'', s)
    
    # Remove all digits (0-9)
    # \d matches any digit character.
    s = re.sub(r"[\d]",'',s)
    
    return s

In [4]:
def preprocessing(words):
    
    tokens = word_tokenize(words)
    tokens_preprocessed = [preprocess_string(token) for token in tokens]
    
    return [word.lower() for word in tokens_preprocessed if len(word) !=0 and
            (word not in string.punctuation)]
        

In [5]:
## tokenized the song
tokens = preprocessing(song)
tokens

['we',
 'are',
 'no',
 'strangers',
 'to',
 'love',
 'you',
 'know',
 'the',
 'rules',
 'and',
 'so',
 'do',
 'i',
 'a',
 'full',
 'commitments',
 'what',
 'im',
 'thinking',
 'of',
 'you',
 'wouldnt',
 'get',
 'this',
 'from',
 'any',
 'other',
 'guy',
 'i',
 'just',
 'wan',
 'na',
 'tell',
 'you',
 'how',
 'im',
 'feeling',
 'got',
 'ta',
 'make',
 'you',
 'understand',
 'never',
 'gon',
 'na',
 'give',
 'you',
 'up',
 'never',
 'gon',
 'na',
 'let',
 'you',
 'down',
 'never',
 'gon',
 'na',
 'run',
 'around',
 'and',
 'desert',
 'you',
 'never',
 'gon',
 'na',
 'make',
 'you',
 'cry',
 'never',
 'gon',
 'na',
 'say',
 'goodbye',
 'never',
 'gon',
 'na',
 'tell',
 'a',
 'lie',
 'and',
 'hurt',
 'you',
 'weve',
 'known',
 'each',
 'other',
 'for',
 'so',
 'long',
 'your',
 'hearts',
 'been',
 'aching',
 'but',
 'youre',
 'too',
 'shy',
 'to',
 'say',
 'it',
 'inside',
 'we',
 'both',
 'know',
 'whats',
 'been',
 'going',
 'on',
 'we',
 'know',
 'the',
 'game',
 'and',
 'were',
 'gon',

### Vocabulary 

In [6]:
vocab = set(tokens)
vocab

{'a',
 'aching',
 'and',
 'any',
 'are',
 'around',
 'ask',
 'been',
 'blind',
 'both',
 'but',
 'commitments',
 'cry',
 'desert',
 'do',
 'dont',
 'down',
 'each',
 'feeling',
 'for',
 'from',
 'full',
 'game',
 'get',
 'give',
 'going',
 'gon',
 'goodbye',
 'got',
 'guy',
 'hearts',
 'how',
 'hurt',
 'i',
 'if',
 'im',
 'inside',
 'it',
 'just',
 'know',
 'known',
 'let',
 'lie',
 'long',
 'love',
 'make',
 'me',
 'na',
 'never',
 'no',
 'of',
 'on',
 'other',
 'play',
 'rules',
 'run',
 'say',
 'see',
 'shy',
 'so',
 'strangers',
 'ta',
 'tell',
 'the',
 'thinking',
 'this',
 'to',
 'too',
 'understand',
 'up',
 'wan',
 'we',
 'were',
 'weve',
 'what',
 'whats',
 'wouldnt',
 'you',
 'your',
 'youre'}

In [7]:
text_pipeline_old = lambda x: vocab(word_tokenize(x))
text_pipeline_old(song)[0:10]


TypeError: 'set' object is not callable

**For this error we can callable sets, because vocab is created using sets instead of build_vocab so we have to define get_tokenizer uisng torchtext**

In [8]:
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
song= """We are no strangers to love
You know the rules and so do I
A full commitments what Im thinking of
You wouldnt get this from any other guy
I just wanna tell you how Im feeling
Gotta make you understand
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Weve known each other for so long
Your hearts been aching but youre too shy to say it
Inside we both know whats been going on
We know the game and were gonna play it
And if you ask me how Im feeling
Dont tell me youre too blind to see
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Weve known each other for so long
Your hearts been aching but youre too shy to say it
Inside we both know whats been going on
We know the game and were gonna play it
I just wanna tell you how Im feeling
Gotta make you understand
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you"""

In [10]:
tokenizer = get_tokenizer("basic_english")
        
vocabulary = build_vocab_from_iterator(map(tokenizer,song.split()),specials=["<unk>"])
vocabulary.set_default_index(vocabulary["<unk>"])

In [11]:
print(vocabulary(tokens)[0:10])

[21, 58, 70, 74, 25, 69, 2, 20, 31, 72]


In [12]:
# set vocabulary index (that converts raw text into indexes)
text_pipeline = lambda x: vocabulary(tokenizer(x))
text_pipeline(song)[0:10]

[21, 58, 70, 74, 25, 69, 2, 20, 31, 72]

In [13]:
index_to_token = vocabulary.get_itos()
index_to_token[58]

'are'

## Embdedding Layer

An embedding layer is a crucial element in natural language processing (NLP) and neural networks designed for sequential data. It serves to convert categorical variables, like words or discrete indexes representing tokens, into continuous vectors. This transformation facilitates training and enables the network to learn meaningful relationships among words.

Let’s consider a different vocabulary:

Vocabulary: {cat, dog, bird, fish}

Each word is assigned a unique index:

Indices: {0, 1, 2, 3}

When using an embedding layer, each index is mapped to a continuous vector (randomly initialized at first):

Vector for index 0 (cat): [0.5, -0.2, 0.1]
Vector for index 1 (dog): [-0.3, 0.9, 0.4]
Vector for index 2 (bird): [0.7, 0.3, -0.6]
Vector for index 3 (fish): [-0.1, 0.2, 0.8]

In [14]:
import torch.nn as nn 

In [15]:

def gen_embeddeing(vocab):
    
    # define number of dimesions
    embeded_dim = 20
    # define vocabulary size
    vocab_size = len(vocab)
    
    # define an embedding layer
    embedding = nn.Embedding(vocab_size, embeded_dim)
    
    return embedding
    


**Embeddings**: Obtain the embedding for the first word with index 0 or 1. Don't forget that you have to convert the input into a tensor. The embeddings are initially initialized randomly, but as the model undergoes training, words with similar meanings gradually come to cluster closer together


In [16]:
embeddings = gen_embeddeing(vocabulary)
embeddings

Embedding(79, 20)

In [17]:
for n in range(2): 
    embedding=embeddings(torch.tensor(n))
    print("word",index_to_token[n])
    print("index",n)
    print( "embedding", embedding)
    print("embedding shape", embedding.shape)

word <unk>
index 0
embedding tensor([ 0.1028,  0.2292,  0.4303, -0.9281, -0.5459, -0.1090, -0.5226, -0.7952,
        -0.6895,  1.2075, -0.5030,  0.2016,  2.8641, -0.8740, -1.0158, -0.9895,
         0.1676, -1.5735, -1.3862, -0.0868], grad_fn=<EmbeddingBackward0>)
embedding shape torch.Size([20])
word gonna
index 1
embedding tensor([-1.7333, -1.6696, -0.4413, -0.2462, -2.3272, -0.1829,  2.8251,  0.1263,
        -1.5434, -2.8616,  0.2616, -1.2284,  0.9370, -0.4824,  1.0602, -0.3895,
         3.0949, -0.8040,  0.7538, -1.3315], grad_fn=<EmbeddingBackward0>)
embedding shape torch.Size([20])


These vectors will serve as inputs for the next layer.
### Generating context-target pairs (n-grams)

Organize words within a variable-size context using the following approach: Each word is denoted by 'i'. 
To establish the context, simply subtract 'j'. The size of the context is determined by the value of``CONTEXT_SIZE``.


In [18]:
context_size = 2

def gen_ngrams(tokens,context_size):
    
    endsize = len(tokens)
    n_grams = [([tokens[i-j-1] for j in range(context_size)],tokens[i])
               for i in range(context_size, endsize)] 
    
    return n_grams

In [21]:
gen_ngrams(tokens,2)

[(['are', 'we'], 'no'),
 (['no', 'are'], 'strangers'),
 (['strangers', 'no'], 'to'),
 (['to', 'strangers'], 'love'),
 (['love', 'to'], 'you'),
 (['you', 'love'], 'know'),
 (['know', 'you'], 'the'),
 (['the', 'know'], 'rules'),
 (['rules', 'the'], 'and'),
 (['and', 'rules'], 'so'),
 (['so', 'and'], 'do'),
 (['do', 'so'], 'i'),
 (['i', 'do'], 'a'),
 (['a', 'i'], 'full'),
 (['full', 'a'], 'commitments'),
 (['commitments', 'full'], 'what'),
 (['what', 'commitments'], 'im'),
 (['im', 'what'], 'thinking'),
 (['thinking', 'im'], 'of'),
 (['of', 'thinking'], 'you'),
 (['you', 'of'], 'wouldnt'),
 (['wouldnt', 'you'], 'get'),
 (['get', 'wouldnt'], 'this'),
 (['this', 'get'], 'from'),
 (['from', 'this'], 'any'),
 (['any', 'from'], 'other'),
 (['other', 'any'], 'guy'),
 (['guy', 'other'], 'i'),
 (['i', 'guy'], 'just'),
 (['just', 'i'], 'wan'),
 (['wan', 'just'], 'na'),
 (['na', 'wan'], 'tell'),
 (['tell', 'na'], 'you'),
 (['you', 'tell'], 'how'),
 (['how', 'you'], 'im'),
 (['im', 'how'], 'feeling'

Output the first element, which results in a tuple. The initial element represents the context, and the index indicates the following word.


In [24]:
ngrams=gen_ngrams(tokens,2)
context, target=ngrams[0]
print("context",context,"target",target)
print("context index",vocabulary(context),"target index",vocabulary([target]))

context ['are', 'we'] target no
context index [58, 21] target index [70]


The Core Reason: Flattening Concatenated Embeddings
In an n-gram model, you have CONTEXT_SIZE number of token embeddings, and each embedding is a vector of size embedding_dim. Before passing them to the linear layer, you concatenate (flatten) all those embeddings into a single 1D vector.

word_1 → [e1, e2, ..., e20]        # 20 values
word_2 → [e1, e2, ..., e20]        # 20 values  
word_3 → [e1, e2, ..., e20]        # 20 values

That's embedding_dim * CONTEXT_SIZE = 20 * 3 = 60 values — which is exactly why the linear layer needs embedding_dim * CONTEXT_SIZE as its input dimension.

### Linear Layer Define

In [25]:
embeded_dims = 20
linear_layer = nn.Linear(embeded_dims * context_size, 128)

In [26]:
my_embdedding = embeddings(torch.tensor(vocabulary(context)))
my_embdedding

tensor([[-1.7604,  0.3528,  0.7286,  0.6090, -2.1408,  0.6578, -0.7206,  0.5740,
         -2.0355,  1.9466,  2.1930,  0.3455, -1.4252, -0.5742, -0.4707, -1.2031,
          0.5153, -1.6778,  0.4795, -1.1828],
        [-0.7281,  1.7607, -0.2878, -1.0442,  0.8950,  1.8304, -0.3665, -0.1279,
         -1.4182, -0.7067, -1.5013,  0.4832,  1.0291, -0.6788, -0.9306, -0.6985,
          0.3651, -0.5594, -0.7994,  0.8812]], grad_fn=<EmbeddingBackward0>)

**We have to flattern the size before feed to the linear layer**

In [28]:
my_embdedding = my_embdedding.reshape(1,-1)
my_embdedding.shape

torch.Size([1, 40])

In [29]:
linear_layer(my_embdedding)

tensor([[-1.0823, -0.3327,  0.8900,  0.6714, -0.3026,  0.6638,  0.0816,  0.9056,
          0.4990, -1.0386, -0.6828, -0.2156,  0.2509, -0.2750, -0.4455,  0.8418,
          0.0815, -0.0818,  0.5111, -0.5569, -0.1149, -0.0661, -1.1768, -0.5866,
          0.2117, -0.2926,  0.4650, -0.7505, -0.1764, -0.4070,  0.2973, -0.6404,
         -0.7294,  0.3216, -0.6091,  1.1197, -0.9512,  0.2555,  0.7415,  0.5332,
          0.7478, -1.1249,  0.7431,  0.2646, -0.9148,  0.1368,  0.7998, -0.7783,
         -0.7996, -0.0762,  0.7367, -0.0568,  0.5774,  0.1152, -0.4445,  0.8552,
         -0.9320,  0.0611, -0.3952,  0.0523, -0.6156,  1.1008,  0.1162, -0.5431,
         -0.3674, -0.2056, -1.4056, -0.0357, -1.5390,  0.0747, -0.0612,  0.1784,
          1.1306, -0.1606,  1.2525, -0.2547, -0.0026,  0.1170, -0.2698, -0.0139,
          0.3837, -0.2855,  0.1258,  0.6344, -1.5705,  0.9145, -0.1285,  0.0690,
         -0.0603, -0.1512, -0.2969, -1.1825,  0.7600,  0.3131,  0.1828,  0.6217,
         -0.5644,  0.0131,  

## **Batch Function Definition**

In [30]:
from torch.utils.data import DataLoader

In [50]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 10
context_size = 3
embeded_dims = 20

def collate_func(batch):
    
    context, target = [],[]
    batch_size = len(batch)
    
    for i in range(context_size, batch_size):
        
        target.append(vocabulary([batch[i]]))
        context.append(vocabulary([batch[i-j-1]for j in range(context_size)])) 
        
        
        
    return torch.tensor(context).to(device), torch.tensor(target).to(device).reshape(-1)

Similarly, it's important to highlight that the size of the last batch could deviate from that of the earlier batches. To tackle this, the approach involves adjusting the final batch to conform to the specified batch size, ensuring it becomes a multiple of the predetermined size. When necessary, you'll employ padding techniques to achieve this harmonization. One approach you'll use is appending the beginning of the song to the end of the batch.


In [51]:
Padding=batch_size-len(tokens)%batch_size
tokens_pad=tokens+tokens[0:Padding]

### **Data Loader**

In [52]:
data_loader = DataLoader(
    tokens_pad, shuffle=True, batch_size=batch_size, collate_fn=collate_func
)

## Multi-class neural network

You have developed a PyTorch class for a multi-class neural network. The network's output is the probability of the next word within a given context. Therefore, the number of classes corresponds to the count of distinct words. The initial layer consists of embeddings, and in addition to the final layer, an extra hidden layer is incorporated.


In [53]:
import torch.nn.functional as F

In [54]:
class NGramLanuageModel (nn.Module):
    
    def __init__(self, vocab_size, context_size, embeded_dims):
        super(NGramLanuageModel,self).__init__()
        
        # initializing class variables
        self.context_size = context_size
        self.embedded_dim = embeded_dims
        
        # define embdedding function
        self.embeddings = nn.Embedding(vocab_size, self.embedded_dim)
        
        # Fully connected hidden layer: Maps the concatenated embeddings to a 128-dimensional space
        self.linear_layer_1 = nn.Linear(self.embedded_dim * self.context_size, 128)
        
        # Output layer: Maps the hidden layer output to vocabulary size (probability distribution over words)
        self.linear_layer_output = nn.Linear(128, vocab_size)
        
    def forward(self, input):
        
        # Convert input word indices into dense vectors using the embedding layer    
        embed = self.embeddings(input) # Shape: (batch_size, context_size, embedding_dim)
        
        # Reshape the embeddings into a single vector per input sample
        embed = torch.reshape(embed,(-1, self.context_size* self.embedded_dim))
        
        # Apply first fully connected layer with ReLU activation
        out_1 = F.relu(self.linear_layer_1(embed))
        
        # Apply second fully connected layer to generate vocabulary-size logits
        out_2 = self.linear_layer_output(out_1)
        
        return out_2 

### Create a Instance of Model

In [55]:
model = NGramLanuageModel(len(vocabulary),context_size,embeded_dims).to(device)

Retrieve Samples from the Dataloader to Model

In [67]:
data_iter = iter(data_loader)
context, target = next(data_iter)
print("context:\n", context)
print("target:\n", target)

model_target = model(context)
print("Model Target:\n", model_target)

context:
 tensor([[15, 20, 47],
        [32, 15, 20],
        [10, 32, 15],
        [19, 10, 32],
        [36, 19, 10],
        [ 0, 36, 19],
        [ 4,  0, 36]])
target:
 tensor([32, 10, 19, 36,  0,  4,  2])
Model Target:
 tensor([[-2.9727e-01,  1.6128e-01, -9.2819e-02,  3.6714e-02, -4.5037e-01,
          4.1825e-01,  4.9178e-02, -1.6447e-01, -5.8275e-02, -1.6550e-01,
         -3.4702e-01, -4.8394e-01,  3.3007e-01, -3.5874e-02, -2.2368e-01,
          6.1355e-01, -5.2333e-02,  5.4565e-02, -5.3868e-01,  1.4076e-01,
         -6.5370e-03,  1.5688e-01, -1.6121e-01,  3.3629e-01, -1.3643e-02,
          2.7482e-01, -4.0306e-01, -2.2119e-01, -3.8830e-01, -6.3911e-02,
          1.3104e-01, -1.1139e-01,  2.8789e-02,  3.6241e-02, -4.1264e-01,
          3.1363e-01,  2.7031e-01,  4.4493e-02,  1.4114e-01,  3.0437e-01,
         -2.3630e-02,  1.5831e-01,  4.5174e-02, -1.8481e-01,  8.2897e-02,
          4.0715e-01, -9.4293e-02, -3.0477e-01,  6.6180e-01, -3.5170e-01,
          4.0369e-01, -1.5979e-01,

While the model remains untrained, analyzing the output can provide us with a clearer understanding. In the output, the first dimension corresponds to the batch size, while the second dimension represents the probability associated with each class.


In [68]:
model_target.shape

torch.Size([7, 79])

Find the index with the highest probability.


In [69]:
predict_index = torch.argmax(model_target,1)
predict_index

tensor([67, 73, 47, 45, 47, 20, 20])

In [70]:
[index_to_token[i.item()] for i in predict_index]

['guy', 'see', 'me', 'known', 'me', 'know', 'know']

Create a function that accomplishes the same task for the tokens.


In [71]:
def write_song(model, my_song, num_of_words = 100):
    
    # Get the mapping from index to word for decoding predictions
    index_to_tokens = vocabulary.get_itos()
    
    # Loop to generate the desired number of words
    for i in range(num_of_words):
        
        with torch.no_grad():
            
            context = torch.tensor(
                vocabulary([tokens[i-j-1] for j in range(context_size)])
            ).to(device)
            
            # Predict the next word by selecting the word with the highest probability
            word_idx = torch.argmax(model(context), 1)
            
            # Append the predicted word to the generated text
            my_song += " " +  index_to_tokens[word_idx.detach().item()]
    
    return my_song 